# Recommendation Engine Project: Collaborative Filtering with Implicit Feedback

## Purpose

This is a **self-directed project** focused on building a recommendation system using collaborative filtering on implicit feedback data. You'll work with real gaming data where the signal isn't star ratings but playtime.

**What you'll practice:**
- Parsing non-standard data formats (gzip'd Python dicts)
- Handling implicit feedback (playtime instead of ratings)
- Building user and item embeddings for ranking
- Negative sampling strategies
- Evaluation with ranking metrics (Precision@K, NDCG) instead of RMSE
- Embedding analysis and visualization

**Why implicit feedback?**

In the lesson notebook, we worked with MovieLens: users explicitly rated movies 1-5 stars. Clean, easy to model. But most real-world recommendation data doesn't look like that. Spotify knows what you listened to, not what you'd rate it. Amazon knows what you bought, not how much you liked it. Steam knows how many hours you played, not whether you enjoyed them.

Implicit feedback is noisier, harder to interpret, and requires different modeling choices. That's exactly why it's worth practicing.

## Dataset Options

| Dataset | Interactions | Signal Type | Format | Difficulty |
|---------|-------------|-------------|--------|------------|
| **MovieLens** | 100K | Explicit (1-5 stars) | Clean CSV | Easy |
| **Food.com** | 1.3M | Explicit ratings + review text | CSV | Medium |
| **Steam Games** | 5.1M | Implicit (playtime minutes) | gzip'd Python dicts | Medium-Hard |

## Dataset Deep Dive

### 1. MovieLens

**Location:** `../../../../data/movielens/ratings.csv`

100K explicit ratings from 610 users across 9,724 movies. Clean CSV with columns: userId, movieId, rating (0.5-5.0), timestamp.

| Pros | Cons |
|------|------|
| Clean, fast, familiar | You already used it in the lesson notebook |
| Explicit ratings are easier to model | Only 100K interactions |
| Well-benchmarked everywhere | Doesn't teach implicit feedback |

### 2. Food.com

**Location:** Download required — see `../../../../data/DATASETS.md` for instructions (`kaggle datasets download shuyangli94/food-com-recipes-and-user-interactions`). Place in `../../../../data/food_recipes/`.

1.1M interactions between users and recipes. Columns: user_id, recipe_id, date, rating (0-5), review text. Also has `RAW_recipes.csv` with rich metadata (ingredients, nutrition, steps, tags).

| Pros | Cons |
|------|------|
| Large (1.1M interactions) | Explicit ratings again |
| Rich metadata for hybrid approaches | Review text is noisy |
| Both rating + text signals available | Recipes are a niche domain |
| Good for content-based experiments | |

### 3. Steam Games

**Location:** Download required — see `../../../../data/DATASETS.md` for instructions (`kaggle datasets download tamber/steam-video-games`). Place in `../../../../data/steam_games/`.

Two files:
- `australian_users_items.json.gz` — 88,310 users, 10,978 games, 5.1M interactions with playtime
- `australian_user_reviews.json.gz` — 25,799 users, 59K reviews with recommend boolean + review text

**Format:** gzip'd file where each line is a Python dict (NOT JSON — uses single quotes, needs `ast.literal_eval`). Nested structure: each line is one user with all their items/reviews as a list.

Items file structure per line:
```python
{
    'user_id': '76561197970982479',
    'items_count': 277,
    'steam_id': '76561197970982479',
    'user_url': 'http://steamcommunity.com/profiles/76561197970982479',
    'items': [
        {'item_id': '10', 'item_name': 'Counter-Strike', 'playtime_forever': 6, 'playtime_2weeks': 0},
        {'item_id': '20', 'item_name': 'Team Fortress Classic', 'playtime_forever': 0, 'playtime_2weeks': 0},
        ...
    ]
}
```

| Pros | Cons |
|------|------|
| Implicit feedback — the real-world paradigm | Non-standard format needs parsing |
| Massive: 5.1M interactions | Playtime is extremely right-skewed |
| Real game names (Counter-Strike, Dota 2) | ~33% of interactions have zero playtime |
| Reviews available for hybrid models | Australian users only (subset) |
| Teaches negative sampling, ranking metrics | |

## Recommendations

### Top Pick: Steam Games

**Why:**
- Different domain from the lesson notebook (avoid copy-pasting MovieLens code)
- Implicit feedback is how most production recommendation systems work (Spotify, Netflix, Amazon)
- Parsing the gzip format is real-world data engineering practice
- Forces you to think about what "positive signal" means when there are no star ratings
- Playtime as signal is interesting: does 500 hours mean they love the game, or that it's addictive?
- Review text available if you want to try hybrid approaches later

### Alternative: Food.com

If you want to focus on hybrid approaches (collaborative + content-based), Food.com has both ratings AND rich recipe metadata (ingredients, nutrition, cooking steps). Good for experimenting with combining embedding-based recommendations with content features.

### Avoid for this project: MovieLens

You already built a full MovieLens pipeline in the lesson notebook. Repeating it here teaches nothing new.

## Your Choice

**Dataset I'm using for this project:**

*(Write your choice here)*

In [ ]:
# Your imports here


# Part A: Implicit Collaborative Filtering

The lesson notebook used explicit ratings (1-5 stars) and optimized for RMSE (how close is the predicted rating to the actual one?). With implicit feedback, the task changes fundamentally: we're not predicting a score, we're learning to **rank** items. Given a user, which items should appear at the top of their recommendation list?

This shift affects every part of the pipeline: how we define positive/negative examples, what loss function we use, and how we measure success.

## Phase A1: Data Exploration

**Task A1.1: Load and parse the data**

For Steam: the items file (`australian_users_items.json.gz`) is gzip-compressed, with one Python dict per line. You'll need to:
1. Open with `gzip.open(..., 'rt')` to read as text
2. Parse each line with `ast.literal_eval` (NOT `json.loads` — the data uses single quotes)
3. Flatten the nested structure: each user has a list of items, and each item becomes a row

Target DataFrame columns: `user_id, item_id, item_name, playtime_forever, playtime_2weeks`

This will take 30-60 seconds for 5.1M rows. Consider building a list of dicts, then converting to DataFrame at the end.

**Task A1.2: Explore the interaction data**

- Shape, dtypes, null values
- How many unique users and unique games?
- Playtime distribution: min, max, mean, median, 25th/75th/99th percentiles
- How many interactions have zero playtime? (User owns the game but never played it)
- What percentage of the full user-game matrix is filled? (Sparsity)

**Task A1.3: Visualize**

1. Playtime distribution histogram (use log scale — the raw distribution is unusable)
2. Top 20 most-played games (by total playtime or by number of users)
3. Games per user: histogram
4. Users per game: histogram
5. Sparsity: compute and display

**Task A1.4: (Optional) Load reviews data**

Parse `australian_user_reviews.json.gz` the same way. 25,799 users with 59K reviews total. Each review has:
- `recommend`: boolean (True/False)
- `review`: text
- `item_id`: which game
- `posted`: date string
- `helpful`: helpfulness votes string

This is additional signal you can use in Part B (hybrid model).

## Phase A2: Preprocessing

**Task A2.1: Define your interaction signal**

Raw `playtime_forever` in minutes is problematic: range is 0 to 551K, distribution is extremely right-skewed. You need to transform it into something the model can work with.

Options:

| Approach | How | Pros | Cons |
|----------|-----|------|------|
| **Binary** | played > 0 → 1, else 0 | Simplest, clean signal | Loses intensity information |
| **Log transform** | `log1p(playtime)` | Preserves ordering, compresses range | Still continuous, need to decide loss |
| **Confidence weighting** | `c = 1 + alpha * log1p(playtime)` | Theoretically grounded (Hu et al. 2008) | More complex implementation |
| **Thresholded** | playtime > X minutes → 1 | Filters noise (owns-but-never-played) | Choosing X is arbitrary |

Think about what makes sense. A user with 0 minutes might mean the game was bundled, gifted, or bought on sale and never played. Is that a positive signal?

**Task A2.2: Filter sparse users and items**

Remove users with fewer than 5 interactions and games with fewer than 5 interactions. This is standard practice — users with 1-2 games give the model almost nothing to learn from, and obscure games with 1-2 players can't develop meaningful embeddings.

Report: how many users, games, and interactions remain after filtering?

**Task A2.3: Create contiguous ID mappings**

After filtering, user_ids and item_ids have gaps. `nn.Embedding` needs contiguous integers starting at 0. Create `user_to_idx` and `item_to_idx` dictionaries, and also reverse mappings (`idx_to_user`, `idx_to_item`) for later interpretation.

Apply the mappings to your DataFrame.

**Task A2.4: Train/validation split**

Two common approaches for implicit data:

1. **Random split**: randomly hold out 20% of interactions. Simple but a user's training and validation items are randomly mixed.
2. **Leave-one-out**: for each user, hold out 1 random interaction for validation. Standard for ranking evaluation (each user has exactly 1 item to "find" in the ranking).

Leave-one-out is more standard for ranking evaluation, but random split is simpler to implement and debug. Choose one and implement it.

**Task A2.5: Negative sampling**

With explicit ratings, every (user, movie, rating) row is a training example. With implicit data, you only have **positive** examples (user played game). You need to generate negatives.

For each positive pair (user, game_played), sample N games the user has NOT interacted with. These become negative examples: (user, game_not_played). Common ratio: 4 negatives per positive.

Important: resample negatives each epoch. If you fix them once, the model memorizes specific negative items instead of learning general patterns.

Think about efficiency: with 5M positives and 4 negatives each, you're creating 25M training pairs per epoch.

## Phase A3: Model Architecture

**Task A3.1: Create PyTorch Dataset**

Two common patterns for implicit feedback:

**Triplet-based (for BPR):** Each sample returns `(user_idx, positive_item_idx, negative_item_idx)`. The loss function pushes the positive item's score above the negative item's score.

**Pair-based (for BCE):** Each sample returns `(user_idx, item_idx, label)` where label is 1 for positive interactions and 0 for sampled negatives. Standard binary classification.

Choose one that matches your loss function choice in A3.2.

**Task A3.2: Build the model**

Two recommended architectures:

**Option 1: BPR (Bayesian Personalized Ranking)**
- User embedding + item embedding
- Score = `dot(user_emb, item_emb)`
- Loss: `-log(sigmoid(score_positive - score_negative))` per triplet
- The model learns that positive items should rank higher than negative ones
- Paper: Rendle et al. 2009

**Option 2: BCE with negative sampling**
- User embedding + item embedding
- Score = `dot(user_emb, item_emb) + user_bias + item_bias`
- Loss: `BCEWithLogitsLoss` treating positive=1, negative=0
- Conceptually simpler, same pattern as binary classification

BPR is the more standard choice for implicit feedback. BCE is easier if you want to get something working quickly.

Start with embedding dimension 64.

**Task A3.3: Verify your model**

- Pass a dummy batch through, check output shapes
- Print parameter count
- With ~88K users and ~11K games at d=64: `(88000 + 11000) * 64 ≈ 6.3M` parameters

**Task A3.4: (Optional) Neural version**

After getting the dot product model working, try concatenating user and item embeddings and passing through an MLP (like the NeuralCollabFilter from the lesson notebook). Compare performance.

## Phase A4: Training

**Task A4.1: Training loop**

Implement the training loop. Key differences from the lesson notebook:
- If using BPR: resample negative items each epoch
- Track training loss per epoch
- Suggested: lr=1e-3, weight_decay=1e-5, 20-30 epochs
- Use GPU — with 5M+ interactions, CPU will be slow

**Task A4.2: Evaluation metrics**

RMSE is meaningless for ranking. Use these instead:

**Precision@K:** Of the top K recommended items for a user, how many are in their validation set? "Out of 10 recommendations, how many did the user actually interact with?"

**Recall@K:** Of all items the user interacted with in validation, how many appear in the top K? "Of the games they actually played, how many did we find?"

**NDCG@K (Normalized Discounted Cumulative Gain):** Like Precision@K but gives more credit when relevant items appear earlier in the list. A hit at position 1 matters more than a hit at position 10.

Compute these at K=10 for a sample of users (computing for all 88K users every epoch is expensive — sample 1000-5000 users).

**Task A4.3: Popularity baseline**

Before celebrating your model's metrics, check: can you beat the dumbest possible strategy?

Recommend the most-played games to every user (same list for everyone). Compute the same Precision@K, Recall@K, NDCG@K. Your model should beat this — if it doesn't, the model hasn't learned anything beyond popularity.

## Phase A5: Evaluation and Analysis

**Task A5.1: Generate recommendations**

Pick 5 users with different play histories (one casual gamer, one hardcore gamer, one with niche tastes, etc). For each:
1. Print their actual top-10 most-played games
2. Generate top-10 recommended games (excluding games they already played)
3. Do the recommendations make sense? If a user plays mostly FPS games, are the recommendations FPS?

**Task A5.2: Analyze game embeddings**

Project game embeddings to 2D using PCA or t-SNE (same technique from the lesson notebook). Do similar games cluster together?

You don't have genre labels, but you do have game names. Spot-check:
- Do FPS games (Counter-Strike, TF2, Call of Duty) cluster?
- Do strategy games cluster separately?
- Do indie games form their own region?

Label some well-known games on the plot.

**Task A5.3: Find similar games**

Build a "games like this" function using cosine similarity on game embeddings. Test with well-known titles:
- Counter-Strike → ?
- Dota 2 → ?
- Team Fortress 2 → ?
- Civilization V → ?

Do the results make intuitive sense?

**Task A5.4: Hyperparameter sweep**

Try different configurations and track Precision@10:

| Parameter | Values to try |
|-----------|---------------|
| Embedding dimension | 16, 32, 64, 128 |
| Negative samples per positive | 1, 4, 8 |
| Learning rate | 1e-2, 1e-3, 1e-4 |

Keep other parameters fixed while sweeping one at a time.

## Phase A6: Reflection

After completing the project, answer these questions:

1. How does implicit feedback change the modeling problem compared to MovieLens explicit ratings? What assumptions are different?

2. Why is negative sampling necessary? What happens if you train only on positive examples?

3. What makes recommendation fundamentally harder than classification? Think about the feature space, evaluation, and feedback loops.

4. How would you handle a brand new user who just created an account? A brand new game that just launched?

5. Your embedding space learned to place similar games near each other. How is this related to how word2vec places similar words near each other? What's the shared principle?

6. If you deployed this system to real users, what could go wrong? Think about feedback loops, popularity bias, and filter bubbles.

In [ ]:
# Your reflection notes here


# Part B: Extensions (Optional)

## B1: Hybrid Model with Reviews

The reviews file has `recommend` (boolean) and `review` (text) for 59K interactions. Ideas:

1. **Use the recommend boolean as additional signal.** Some users played a game 100 hours but DON'T recommend it (it's addictive but frustrating). Combine playtime and recommend for a richer signal.

2. **Sentiment analysis on review text.** Use a pretrained sentiment model (HuggingFace) to score reviews. Do sentiment scores correlate with playtime? Can they improve recommendations?

3. **Content-based hybrid.** Extract features from review text (TF-IDF or sentence embeddings) and combine with collaborative filtering embeddings. Does the hybrid model handle cold-start better?

## B2: Temporal Analysis

The items file has `playtime_2weeks` alongside `playtime_forever`. This tells you about recent engagement vs historical.

Ideas:
1. Do recent preferences predict better than all-time? Train on `playtime_2weeks` vs `playtime_forever` and compare.
2. Can you detect games a user has abandoned? (High total playtime, zero recent → they stopped playing)
3. Is there a seasonal effect? Games with high recent-to-total ratios might be trending.

## B3: Cross-Domain Transfer

You trained game embeddings that capture similarity. Could these transfer to a new context?

- If a new gaming platform launched with zero play data, could your Steam embeddings bootstrap their recommendation system?
- This connects to **transfer learning** — the same idea behind using pretrained image models (ImageNet → your task) or pretrained language models (GPT → your chatbot).
- Experiment: freeze your game embeddings and use them as fixed features in a different model.

## Reference Materials

**Papers:**
- Hu et al. 2008 — [Collaborative Filtering for Implicit Feedback Datasets](http://yifanhu.net/PUB/cf.pdf) (the WMF paper)
- Rendle et al. 2009 — [BPR: Bayesian Personalized Ranking from Implicit Feedback](https://arxiv.org/abs/1205.2618)
- Koren et al. 2009 — [Matrix Factorization Techniques for Recommender Systems](https://datajobs.com/data-science-repo/Recommender-Systems-[Netflix].pdf) (Netflix Prize)

**PyTorch:**
- [torch.nn.Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
- [torch.nn.functional.cosine_similarity](https://pytorch.org/docs/stable/generated/torch.nn.functional.cosine_similarity.html)

**Evaluation:**
- [Precision and Recall at K](https://en.wikipedia.org/wiki/Evaluation_measures_(information_retrieval))
- [NDCG explained](https://towardsdatascience.com/demystifying-ndcg-bee3be58cfe0)

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>